# Lab 13 - Semi-supervised learning


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelPropagation, LabelSpreading, SelfTrainingClassifier
from sklearn.svm import SVC


In [ ]:
X, y = make_moons(n_samples=1200, noise=0.25, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, stratify=y, random_state=RANDOM_STATE)

plt.scatter(X[:, 0], X[:, 1], c=y, s=10, cmap="coolwarm")
plt.title("Synthetic binary dataset")
plt.show()


In [ ]:
def mask_labels(y, g, random_state=42):
    local_rng = np.random.default_rng(random_state)
    y_ssl = np.full_like(y, fill_value=-1)
    labeled_idx = []
    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        take = max(1, g // len(np.unique(y)))
        labeled_idx.extend(local_rng.choice(cls_idx, size=take, replace=False))
    labeled_idx = np.array(labeled_idx)
    y_ssl[labeled_idx] = y[labeled_idx]
    return y_ssl, labeled_idx


def score_predictions(name, y_true, y_pred, score):
    return {
        "method": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, score),
    }


def evaluate_ssl(g):
    y_ssl, labeled_idx = mask_labels(y_train, g, random_state=RANDOM_STATE + g)
    svc = make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, gamma="scale", random_state=RANDOM_STATE))

    methods = {}
    methods["naive"] = svc.fit(X_train[labeled_idx], y_train[labeled_idx])
    methods["self_training"] = SelfTrainingClassifier(
        make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, gamma="scale", random_state=RANDOM_STATE))
    ).fit(X_train, y_ssl)
    methods["label_propagation"] = LabelPropagation(kernel="rbf", gamma=20).fit(X_train, y_ssl)
    methods["label_spreading"] = LabelSpreading(kernel="rbf", gamma=20, alpha=0.2).fit(X_train, y_ssl)

    rows = []
    for name, model in methods.items():
        pred = model.predict(X_test)
        if hasattr(model, "predict_proba"):
            score = model.predict_proba(X_test)[:, 1]
        else:
            score = model.decision_function(X_test)
        rows.append({"g": g, **score_predictions(name, y_test, pred, score)})
    return rows


results = []
for g in [10, 20, 50, 100, 200]:
    results.extend(evaluate_ssl(g))
results


In [ ]:
for method in sorted({row["method"] for row in results}):
    xs = [row["g"] for row in results if row["method"] == method]
    ys = [row["balanced_accuracy"] for row in results if row["method"] == method]
    plt.plot(xs, ys, marker="o", label=method)
plt.xlabel("number of labeled training observations")
plt.ylabel("balanced accuracy")
plt.grid(True)
plt.legend()
plt.show()


With very few labels, SSL can outperform the naive supervised model because it uses the geometry of unlabeled data. As `g` grows, the naive SVC catches up and the benefit of SSL usually decreases.
